# Maternal Mortality Prediction with XGBoost

This notebook is structured into separate sections: Data Loading, Data Preparation, Model Building, Model Evaluation, Feature Analysis, and Data Visualization.

## Data Loading

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor

sns.set_theme(style='whitegrid')

data_path = 'data/Maternal_Mortality.csv'
df_raw = pd.read_csv(data_path)

print(f'Dataset shape: {df_raw.shape}')
display(df_raw.head())

## Data Preparation

In [ ]:
year_pattern = re.compile(r'\((19|20)\d{2}\)$')
year_cols = [c for c in df_raw.columns if year_pattern.search(c)]

undp_col = next((c for c in df_raw.columns if 'UNDP' in c), None)
if undp_col is None:
    raise ValueError('Could not find UNDP region column in dataset.')

id_cols = [
    'ISO3',
    'Country',
    'Continent',
    'Hemisphere',
    'Human Development Groups',
    undp_col,
    'HDI Rank (2021)'
]

df_long = df_raw.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name='MMR_Year_Column',
    value_name='MMR'
)

df_long['Year'] = df_long['MMR_Year_Column'].str.extract(r'(\d{4})').astype(int)
df_long['MMR'] = pd.to_numeric(df_long['MMR'], errors='coerce')
df_long['HDI Rank (2021)'] = pd.to_numeric(df_long['HDI Rank (2021)'], errors='coerce')

df_long['Human Development Groups'] = df_long['Human Development Groups'].fillna('Unknown')
df_long['Continent'] = df_long['Continent'].fillna('Unknown')
df_long[undp_col] = df_long[undp_col].fillna('Unknown')
df_long['HDI Rank (2021)'] = df_long['HDI Rank (2021)'].fillna(df_long['HDI Rank (2021)'].median())

hdi_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Very High': 3}
df_long['hdi_group_enc'] = df_long['Human Development Groups'].map(hdi_map).fillna(1).astype(int)

continent_encoder = LabelEncoder()
undp_encoder = LabelEncoder()
df_long['continent_enc'] = continent_encoder.fit_transform(df_long['Continent'].astype(str))
df_long['undp_enc'] = undp_encoder.fit_transform(df_long[undp_col].astype(str))

df_long = df_long.sort_values(['ISO3', 'Year']).reset_index(drop=True)

df_long['lag_1'] = df_long.groupby('ISO3')['MMR'].shift(1)
df_long['lag_2'] = df_long.groupby('ISO3')['MMR'].shift(2)
df_long['lag_3'] = df_long.groupby('ISO3')['MMR'].shift(3)
df_long['rolling_mean_5'] = (
    df_long.groupby('ISO3')['MMR']
    .shift(1)
    .rolling(window=5, min_periods=1)
    .mean()
)
df_long['mmr_change'] = df_long.groupby('ISO3')['MMR'].diff()
df_long['year_norm'] = (df_long['Year'] - df_long['Year'].min()) / (df_long['Year'].max() - df_long['Year'].min())

model_df = df_long.dropna(subset=['MMR', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_5', 'mmr_change']).copy()

print(f'Prepared dataset shape: {model_df.shape}')
display(model_df[['ISO3', 'Country', 'Year', 'MMR', 'lag_1', 'rolling_mean_5']].head())

## Model Building (XGBoost)

In [ ]:
feature_cols = [
    'Year',
    'HDI Rank (2021)',
    'lag_1',
    'lag_2',
    'lag_3',
    'rolling_mean_5',
    'mmr_change',
    'year_norm',
    'continent_enc',
    'hdi_group_enc',
    'undp_enc'
]

target_col = 'MMR'

train_df = model_df[model_df['Year'] <= 2017].copy()
test_df = model_df[model_df['Year'] > 2017].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror'
)

xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

print(f'Training rows: {len(train_df):,}')
print(f'Testing rows: {len(test_df):,}')

## Model Evaluation

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('Evaluation Metrics')
print(f'MAE : {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R2  : {r2:.4f}')

plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
plt.xlabel('Actual MMR')
plt.ylabel('Predicted MMR')
plt.title('Actual vs Predicted MMR (XGBoost)')
plt.show()

## Feature Analysis

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

display(importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## Data Visualization

In [ ]:
viz_df = df_long.dropna(subset=['MMR']).copy()

global_yearly = viz_df.groupby('Year', as_index=False)['MMR'].mean()
continent_yearly = viz_df.groupby(['Year', 'Continent'], as_index=False)['MMR'].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.lineplot(data=global_yearly, x='Year', y='MMR', ax=axes[0], color='tab:blue')
axes[0].set_title('Global Average MMR Trend')
axes[0].set_ylabel('Average MMR')

sns.lineplot(data=continent_yearly, x='Year', y='MMR', hue='Continent', ax=axes[1])
axes[1].set_title('MMR Trend by Continent')
axes[1].set_ylabel('Average MMR')
axes[1].legend(title='Continent', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()